# Lab 24-01 · Loss, Stubs and Crosstalk

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import si


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Conductor against dielectric loss

*Predict* where FR-4's dielectric loss overtakes a 5-mil, 50 Ω trace's conductor loss.

In [2]:
f = np.logspace(8, np.log10(60e9), 400)
w = 5 * si.MIL
cond = np.array([si.conductor_loss_db_per_in(x, w, 50) for x in f])
fr4 = np.array([si.dielectric_loss_db_per_in(x, 4.4, 0.014) for x in f])
m6 = np.array([si.dielectric_loss_db_per_in(x, 3.4, 0.002) for x in f])
cross = (si.conductor_loss_db_per_in(1e9, w, 50) / si.dielectric_loss_db_per_in(1e9, 4.4, 0.014)) ** 2 * 1e9
print(f"FR-4 dielectric loss overtakes conductor loss at {cross/1e9:.2f} GHz (sqrt(f) against f)")
for fx in (8e9, 16e9, 53.125e9):
    print(f"{fx/1e9:7.3f} GHz: conductor {si.conductor_loss_db_per_in(fx, w, 50):.3f}, "
          f"FR-4 diel {si.dielectric_loss_db_per_in(fx, 4.4, 0.014):.3f}, "
          f"Megtron 6 diel {si.dielectric_loss_db_per_in(fx, 3.4, 0.002):.3f} dB/in")
plt.figure()
plt.loglog(f / 1e9, cond, label="conductor (5 mil, 50 ohm)")
plt.loglog(f / 1e9, fr4, label="FR-4 dielectric")
plt.loglog(f / 1e9, m6, label="Megtron 6 dielectric")
plt.xlabel("frequency (GHz)"); plt.ylabel("loss (dB/in)"); plt.legend(); plt.grid(True, which="both")
plt.show()

FR-4 dielectric loss overtakes conductor loss at 4.46 GHz (sqrt(f) against f)
  8.000 GHz: conductor 0.405, FR-4 diel 0.543, Megtron 6 diel 0.068 dB/in
 16.000 GHz: conductor 0.573, FR-4 diel 1.086, Megtron 6 diel 0.136 dB/in
 53.125 GHz: conductor 1.045, FR-4 diel 3.607, Megtron 6 diel 0.453 dB/in


## Part 2 · Pulse through 10 inches

*Predict* which laminate keeps more of a single 31.25 ps pulse's height after 10 inches.

In [3]:
for name, er, td in (("FR-4", 4.4, 0.014), ("Megtron 6", 3.4, 0.002)):
    h = lambda ff, er=er, td=td: si.s21(si.line_abcd(ff, 10 * si.INCH, 50, er, td, w))
    t, y = si.pulse_response(h, 31.25e-12)
    if name == "FR-4":
        plt.figure()
    print(f"{name:9s}: peak {y.max():.3f} of the 1 V pulse, at {t[y.argmax()]*1e9:.3f} ns")
    plt.plot(t * 1e9, y, label=name)
plt.xlim(1.4, 2.2); plt.xlabel("time (ns)"); plt.ylabel("received (V)"); plt.legend(); plt.grid(True)
plt.show()

FR-4     : peak 0.390 of the 1 V pulse, at 1.793 ns
Megtron 6: peak 0.616 of the 1 V pulse, at 1.578 ns


## Part 3 · Via stubs

*Predict* the notch of a 1.6 mm stub and of a 0.4 mm back-drilled stub, then compare the analytic model with ngspice.

In [4]:
f = np.linspace(1e9, 120e9, 6000)
plt.figure()
for length in (1.6e-3, 0.4e-3):
    thru = si.line_abcd(f, 1e-6, 50, 3.6, 0.0, 1.0)
    m = si.stub_abcd(f, length, 50, 3.6)
    s = np.abs(si.s21(np.einsum("nij,njk->nik", thru, m)))
    print(f"{length*1e3:.1f} mm stub: analytic notch {si.stub_notch(length, 3.6)/1e9:.2f} GHz; "
          f"numerical minimum {f[np.argmin(s)]/1e9:.2f} GHz")
    plt.plot(f / 1e9, 20 * np.log10(s), label=f"{length*1e3:.1f} mm stub")
sim = ngspice("via_stub.cir")
print(f"ngspice notches: {sim['fnotch1']/1e9:.2f} and {sim['fnotch2']/1e9:.2f} GHz")
plt.ylim(-40, 1); plt.xlabel("frequency (GHz)"); plt.ylabel("|S21| (dB)"); plt.legend(); plt.grid(True)
plt.show()

1.6 mm stub: analytic notch 24.69 GHz; numerical minimum 24.68 GHz
0.4 mm stub: analytic notch 98.76 GHz; numerical minimum 98.75 GHz
ngspice notches: 24.69 and 98.76 GHz


## Part 4 · Crosstalk

*Predict* the NEXT plateau and the FEXT area for Example 24-01.7, then what doubling the rise time does to each.

In [5]:
v = si.C0 / np.sqrt(3.0)
for t_r in (30e-12, 60e-12):
    x = si.crosstalk(0.04, 0.06, 50e-3, v, 0.4, t_r)
    print(f"t_r = {t_r*1e12:.0f} ps: NEXT {x['next']*1e3:.2f} mV for {x['next_duration']*1e12:.0f} ps; "
          f"FEXT peak {x['fext']*1e3:.2f} mV; FEXT area {x['fext_area']*1e12:.4f} ps*V")
sim = ngspice("crosstalk.cir")
print(f"ngspice (30 ps): NEXT {sim['next']*1e3:.3f} mV, FEXT area {sim['farea']*1e12:.4f} ps*V, "
      f"FEXT peak {sim['fext']*1e3:.1f} mV (the ladder rings at the ramp's corners)")

t_r = 30 ps: NEXT 10.00 mV for 578 ps; FEXT peak -38.52 mV; FEXT area -1.1555 ps*V
t_r = 60 ps: NEXT 10.00 mV for 578 ps; FEXT peak -19.26 mV; FEXT area -1.1555 ps*V


ngspice (30 ps): NEXT 10.007 mV, FEXT area -1.1554 ps*V, FEXT peak -47.1 mV (the ladder rings at the ramp's corners)
